# Inductive Bias & Equivariance — from scratch

> Lesson: [Inductive Bias & Equivariance](https://ml-viz-ruby.vercel.app/wiki/inductive-bias)
> · Copy to Drive to run and edit.

Four architectures — CNN, RNN, transformer, GNN — are one idea: **match the model's
symmetry to the data's symmetry**. Here we *measure* two of those symmetries directly:

1. a convolution is **translation-equivariant** — shift the image, every feature shifts;
2. graph message passing is **permutation-equivariant/invariant** — relabel the nodes,
   the answer follows (or is unchanged).

No deep-learning framework — just NumPy, so the symmetry is visible in the arithmetic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.facecolor": "#0f1117", "axes.facecolor": "#0f1117",
    "savefig.facecolor": "#0f1117", "text.color": "#e2e8f0",
    "axes.labelcolor": "#e2e8f0", "xtick.color": "#94a3b8", "ytick.color": "#94a3b8",
    "axes.edgecolor": "#2e3347", "figure.figsize": (8, 3),
})
rng = np.random.default_rng(0)

## 1 · Convolution is translation-equivariant

Equivariance means $f(g\cdot x) = g\cdot f(x)$: applying the shift $g$ then the layer
$f$ equals applying $f$ then $g$. We'll convolve a 1-D signal with a small kernel, then
check that **shift-then-convolve == convolve-then-shift**.

In [ ]:
def conv1d_valid(x, k):
    # simple cross-correlation with 'same' length via circular padding (so shifts are clean)
    n, m = len(x), len(k)
    xp = np.concatenate([x[-(m//2):], x, x[:m//2]])         # circular pad
    return np.array([xp[i:i+m] @ k for i in range(n)])

def roll(x, s):  # the group action g: a circular shift by s
    return np.roll(x, s)

x = rng.normal(size=16)
k = np.array([-1.0, 0.0, 1.0])   # an edge detector
s = 3

lhs = conv1d_valid(roll(x, s), k)   # f(g . x)
rhs = roll(conv1d_valid(x, k), s)   # g . f(x)
print("max |f(shift(x)) - shift(f(x))| =", np.max(np.abs(lhs - rhs)))
assert np.allclose(lhs, rhs), "convolution should be shift-equivariant"
print("convolution is translation-equivariant  ✓")

**What to notice.** The two are equal to floating-point precision. That equality is
*exactly* the weight-sharing bias: one kernel is applied identically at every position,
so a shift of the input can only shift the output. An MLP with independent weights per
position has no such guarantee — it would have to *learn* it from data.

In [ ]:
fig, ax = plt.subplots()
ax.plot(lhs, color="#6366f1", lw=2, label="f(shift(x))")
ax.plot(rhs, color="#14b8a6", lw=2, ls="--", label="shift(f(x))")
ax.set_title("convolution: shift-then-filter == filter-then-shift", color="#e2e8f0")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 2 · Message passing is permutation-equivariant

A GNN layer updates each node from its neighbours: $H' = \sigma(\hat A H W)$ with $\hat A$
the (normalised) adjacency. Relabel the nodes with a permutation matrix $P$ — the graph
is unchanged, just renumbered — and the layer should give the **same features, permuted
the same way**: $f(P\cdot \text{graph}) = P\cdot f(\text{graph})$.

In [ ]:
def gnn_layer(A, H, W):
    A_hat = A + np.eye(len(A))                 # add self-loops
    d = A_hat.sum(1, keepdims=True)
    A_norm = A_hat / d                          # row-normalised aggregation
    return np.maximum(A_norm @ H @ W, 0)        # ReLU(A_norm H W)

# A little graph: 5 nodes, random symmetric adjacency, random node features.
N = 5
A = (rng.random((N, N)) < 0.4).astype(float)
A = np.triu(A, 1); A = A + A.T                  # symmetric, no self-loops
H = rng.normal(size=(N, 3))
W = rng.normal(size=(3, 2))

# A random permutation of the node labels.
perm = rng.permutation(N)
P = np.eye(N)[perm]

out_then_perm = P @ gnn_layer(A, H, W)                       # g . f(graph)
perm_then_out = gnn_layer(P @ A @ P.T, P @ H, W)             # f(g . graph)
print("max |f(perm(graph)) - perm(f(graph))| =",
      np.max(np.abs(perm_then_out - out_then_perm)))
assert np.allclose(out_then_perm, perm_then_out), "GNN layer must be permutation-equivariant"
print("message passing is permutation-equivariant  ✓")

### From equivariance to invariance: the readout

Node-level features are permutation-*equivariant*. A **graph-level** prediction must be
permutation-*invariant* — relabelling nodes can't change "is this molecule toxic?". You
get invariance by pooling over nodes with a symmetric function (sum/mean/max).

In [ ]:
def graph_readout(A, H, W):
    return gnn_layer(A, H, W).sum(0)            # sum-pool over nodes -> invariant

orig = graph_readout(A, H, W)
perm = graph_readout(P @ A @ P.T, P @ H, W)
print("readout(original) :", np.round(orig, 4))
print("readout(permuted) :", np.round(perm, 4))
assert np.allclose(orig, perm), "sum-pooled readout must be permutation-invariant"
print("sum-pool readout is permutation-invariant  ✓  (equivariant layers + symmetric pool)")

**What to notice.** Same idea as the CNN's global-average-pool head: keep
**equivariance** through the layers (so structure survives), then collapse to
**invariance** at the very end (so the label ignores relabelling). Break that order —
pool too early, or use a non-symmetric readout — and you destroy the bias you built.

## ✏️ Your turn — build the permutation *invariance* the wrong way

A `max`-over-nodes readout is invariant; an *indexing* readout (`H[0]`, "the first
node") is **not** — it depends on the labelling. Implement both and show which one
survives a permutation.

In [ ]:
def readout_max(A, H, W):
    # TODO(you): return a permutation-INVARIANT summary (hint: a symmetric reduction)
    feats = gnn_layer(A, H, W)
    return ...   # replace ...

def readout_first(A, H, W):
    # already permutation-DEPENDENT: it looks at whichever node is labelled 0
    return gnn_layer(A, H, W)[0]

In [ ]:
inv_ok  = np.allclose(readout_max(A, H, W), readout_max(P @ A @ P.T, P @ H, W))
dep_same = np.allclose(readout_first(A, H, W), readout_first(P @ A @ P.T, P @ H, W))
assert inv_ok, "readout_max must be permutation-invariant"
assert not dep_same, "readout_first should (in general) change under a permutation"
print("max-pool invariant:", inv_ok, "| first-node readout invariant:", dep_same)
print("correct: a symmetric reduction is invariant; indexing a label is not")

<details>
<summary>Solution</summary>

```python
def readout_max(A, H, W):
    return gnn_layer(A, H, W).max(0)   # max over nodes is a symmetric reduction
```

`max`, `sum`, and `mean` over the node axis are all permutation-invariant because they
don't depend on the order. Anything that indexes a specific label (`[0]`, "the node with
the largest id") is not — it rides on the arbitrary numbering, which is precisely the
symmetry a GNN is supposed to ignore.
</details>

## Key takeaways

- **Equivariance** ($f(g\cdot x)=g\cdot f(x)$) is the mechanism behind CNNs, RNNs,
  transformers, and GNNs — each ties weights across one symmetry group.
- The right bias is **weight sharing across a true symmetry**; it buys sample efficiency
  by learning a pattern once and reusing it everywhere the symmetry allows.
- Nets stay **equivariant** through depth and become **invariant** only at the readout —
  get that order wrong and you throw the bias away.
- A transformer assumes the *least* (only permutation) — hence positional encodings, and
  hence its generality when data is abundant.